In [2]:
import matplotlib.pyplot as plt
import pandas as pd
import os
import evaluate
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

In [93]:
dataset = load_dataset("clapAI/MultiLingualSentiment")

In [94]:
train_ds = dataset["train"].shuffle(seed=42).select(range(1000))
train_ds

Dataset({
    features: ['text', 'label', 'source', 'domain', 'language'],
    num_rows: 1000
})

In [96]:
labels = train_ds.unique("label")
label2id = {label: idx for idx, label in enumerate(labels)}
id2label = {idx: label for idx, label in enumerate(labels)}
num_labels = len(labels)
labels

Flattening the indices:   0%|          | 0/1000 [00:00<?, ? examples/s]

['neutral', 'positive', 'negative']

In [84]:
model_checkpoint = "distilbert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [115]:
def preprocess_function(examples):
    # 分词，最大长度设为256（数据集最长文本长度约为160）
    tokenized = tokenizer(examples["text"], truncation=True, max_length=256)
    tokenized["label"] = [label2id[l] for l in examples["label"]]
    return tokenized


tokenized_datasets = train_ds.map(preprocess_function, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(["token_type_ids", "source", "domain"])
print(tokenized_datasets[:2])

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

{'text': ['반전은 좋은데 그러네 ㅎㅎ', 'Очень сильно пахнет. Нет, воняет. Плюс прислали другой размер. Деньги вернули.'], 'label': [0, 0], 'language': ['ko', 'ru'], 'input_ids': [[101, 9321, 16617, 10892, 79633, 28911, 8924, 30873, 77884, 100, 102], [101, 523, 63549, 36341, 12634, 10353, 50800, 119, 21124, 10351, 117, 10439, 54907, 119, 524, 104782, 10913, 37936, 10783, 28698, 66799, 119, 54640, 14122, 543, 80369, 30102, 119, 102]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}


In [113]:
tks = tokenizer.convert_ids_to_tokens(tokenized_datasets[3]["input_ids"])
print(tks)

['[CLS]', 'Nice', 'co', '##ckt', '##ail', '##s', 'and', 'great', 'service', 'from', 'att', '##enti', '##ve', 'staff', '.', '[SEP]']


In [116]:
print(tokenizer.convert_tokens_to_string(tks))

[CLS] Nice cocktails and great service from attentive staff. [SEP]
